# Chapter 9: Claim Frequency and Severity Distributions

Code of Chapter 9 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

The walkthrough comes first, because it loads the data and imports used by the other examples of the chapter.

## 9.7 Python Walkthrough

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from gidata import synthetic_portfolio
from gilossmodels import (claim_counts, count_table, poisson_fit, negbin_moments,
                          negbin_pmf, ab0_ratios, expected_counts, chi_square,
                          fit_moments, fit_mle, frozen, loglik, fit_summary,
                          mean_excess, lognormal_mean_excess, top_share)

policies, claims = synthetic_portfolio()          # the Chapter 2 portfolio, seed 2025

# Section 9.1: counts per policy
n = claim_counts(policies, claims)
t = count_table(n)                                # 4531, 453, 16
lam = poisson_fit(n)                              # 0.097
chi_square(expected_counts(t, lambda k: stats.poisson.pmf(k, lam)), 1)   # (2.05, 1, 0.15)

# Section 9.1.5: the fleet
f = pd.Series([860, 110, 22, 6, 2])
c = f.index.repeat(f.values).to_series()
r, p = negbin_moments(c)                          # 0.4509, 0.7147
chi_square(expected_counts(f, lambda k: negbin_pmf(k, r, p)), 2)        # (0.15, 1, 0.70)

# Section 9.3.3: five claims
s = np.array([500, 1200, 2000, 3500, 7800.0])
fit_moments(s, "lognormal"), fit_mle(s, "lognormal")   # (7.7264, 0.7483), (7.6056, 0.9325)
frozen("lognormal", fit_mle(s, "lognormal")).sf(10_000)  # 0.0426

# Section 9.3.5 and 9.4: the 2025 claims
x = claims["ultimate_amount"].to_numpy()
fit_summary(x)[["loglik", "aic", "ks", "mean"]].round(2)
mean_excess(x, [1000, 2500, 5000, 10000, 20000]).round(2)
top_share(x, 0.01), top_share(x, 0.05), top_share(x, 0.10)   # 0.087, 0.283, 0.436

## Exercise 1: Poisson or negative binomial?

In [ ]:
g = pd.Series([700, 220, 60, 15, 5])
c = g.index.repeat(g.values).to_series()
r, p = negbin_moments(c)                                    # 1.5403, 0.7918
chi_square(expected_counts(g, lambda k: negbin_pmf(k, r, p)), 2)   # (0.135, 1, 0.71)

## Exercise 3: How many large claims?

In [ ]:
fs = fit_summary(x)
[485 * frozen(d, fs.loc[d, "params"]).sf(20_000) for d in fs.index]
# [2.78, 3.68, 12.48, 9.61, 5.06]   exponential, gamma, lognormal, Pareto, Weibull